# Local SDK — Agent Primitives
## FunctionAgent, LLMAgent, RouterAgent, AggregatorAgent, FilterAgent, TransformAgent

---

This notebook is a comprehensive tour of every built-in agent type in the **Multigen SDK**.
By the end you will understand:

- What the `BaseAgent` contract looks like and why it exists
- How to build agents from plain Python functions (sync or async)
- How to use the `@agent` decorator as a shortcut
- How to inspect runtime statistics on any agent
- How `FilterAgent`, `TransformAgent`, `RouterAgent`, and `AggregatorAgent` layer on top
- How `LLMAgent` fits into the same interface
- How to compose agents without any framework machinery

All examples run **entirely locally** — no API keys are needed until the LLMAgent section, where a mock is used.

## 1. Overview — What Are Agents? The `BaseAgent` Abstract Contract

An **agent** in Multigen is the smallest unit of computation. Every agent, regardless of its
internal complexity, satisfies one contract:

```
async def run(context: dict) -> dict
```

- **Input**: a plain Python `dict` called the *context*. It is the shared state that flows through
  a workflow. Agents can read any key from it and add new keys to it.
- **Output**: a plain Python `dict` — a *delta* that gets merged back into the context by the
  orchestrator (Chain, Graph, etc.).

This single-method contract is why agents compose so naturally. A Chain is just a loop that calls
`agent.run(ctx)` in sequence. A Parallel is just a `asyncio.gather` of multiple `.run()` calls.
Nothing more.

### Why `async`?

LLM calls, HTTP requests, and database queries are all I/O-bound. If agents were synchronous,
running ten agents "in parallel" would actually block the thread. By making the interface `async`,
Multigen lets the event loop multiplex thousands of concurrent agent calls on a single thread.
Synchronous functions are automatically wrapped so you can still use them.

### The `BaseAgent` abstract class (simplified)

```python
class BaseAgent(ABC):
    name: str
    stats: AgentStats  # call count, error count, latency histogram

    @abstractmethod
    async def run(self, context: dict) -> dict:
        ...

    async def __call__(self, context: dict) -> dict:
        # Wraps run() with stats tracking, error handling, retries
        ...
```

Calling `agent(ctx)` is identical to `agent.run(ctx)` from the outside, but the `__call__` wrapper
records timing and increments the stats counters automatically.

In [ ]:
# Standard library imports needed for async execution inside Jupyter
import asyncio
import time

# Core agent primitives from the Multigen SDK
from multigen import (
    FunctionAgent,
    RouterAgent,
    AggregatorAgent,
    FilterAgent,
    TransformAgent,
    agent,           # the @agent decorator
)

print("Multigen agent primitives imported successfully.")

## 2. FunctionAgent — Wrapping Plain Python Functions

`FunctionAgent` is the most fundamental agent. It takes any callable — a lambda, a named function,
or an async coroutine function — and wraps it in the `BaseAgent` interface.

This is important because it means you never have to subclass anything to create an agent.
Your existing Python functions become first-class citizens in any Multigen workflow.

### Constructor signature

```python
FunctionAgent(
    fn: Callable[[dict], dict],
    name: str = None,          # auto-derived from fn.__name__ if omitted
    description: str = None,   # used by RouterAgent for semantic routing
)
```

The function receives the full context dict and must return a dict (the delta to merge).
Returning `{}` is valid — it means "I ran but produced no new keys".

Below we show all three function styles: **lambda**, **named sync**, and **async**.

In [ ]:
# ── Style 1: Lambda ──────────────────────────────────────────────────────────
# Lambdas are fine for trivial one-liners. The name defaults to '<lambda>'
# so always pass a name= when using lambdas to keep logs readable.
greet_agent = FunctionAgent(
    fn=lambda ctx: {"greeting": f"Hello, {ctx.get('user', 'stranger')}!"},
    name="greet",
    description="Produces a personalised greeting string.",
)

# Run it — agents are async so we use asyncio.run() in scripts or await in notebooks
result = await greet_agent.run({"user": "Alice"})
print("Lambda agent output:", result)
# Expected: {'greeting': 'Hello, Alice!'}

In [ ]:
# ── Style 2: Named Synchronous Function ──────────────────────────────────────
# Sync functions work just as well. FunctionAgent wraps them with
# asyncio.get_event_loop().run_in_executor() so they don't block the event loop.

def word_count(ctx: dict) -> dict:
    """Count words in ctx['text'] and return the count."""
    text = ctx.get("text", "")
    words = text.split()
    return {"word_count": len(words)}

wc_agent = FunctionAgent(fn=word_count, name="word_counter")

result = await wc_agent.run({"text": "The quick brown fox jumps over the lazy dog"})
print("Sync named function agent output:", result)
# Expected: {'word_count': 9}

In [ ]:
# ── Style 3: Async Function ───────────────────────────────────────────────────
# Async functions are the preferred style when your agent does I/O (HTTP, DB, etc.).
# Here we simulate a slow external fetch with asyncio.sleep.

async def fetch_sentiment(ctx: dict) -> dict:
    """Simulate an async sentiment analysis API call."""
    text = ctx.get("text", "")
    # Simulate network latency without actually calling any API
    await asyncio.sleep(0.05)
    # Naive mock: positive if 'good' or 'great' appears
    sentiment = "positive" if any(w in text.lower() for w in ["good", "great", "love"]) else "neutral"
    return {"sentiment": sentiment}

sentiment_agent = FunctionAgent(fn=fetch_sentiment, name="sentiment_analyzer")

result = await sentiment_agent.run({"text": "I love this great product!"})
print("Async function agent output:", result)
# Expected: {'sentiment': 'positive'}

## 3. The `@agent` Decorator — Convert Any Async Function to a FunctionAgent

Writing `FunctionAgent(fn=my_func, name="my_func")` is repetitive. The `@agent` decorator
does exactly that automatically: it replaces your function definition with an equivalent
`FunctionAgent` instance whose name is taken from the function's `__name__`.

This makes the code feel declarative — you're defining *what the agent does*, not wiring up
wrapper objects by hand. The decorated name still refers to a `FunctionAgent` instance, so
you can call `my_func(ctx)` or `await my_func.run(ctx)` interchangeably.

```python
@agent
async def my_func(ctx: dict) -> dict:
    ...

# Equivalent to:
# my_func = FunctionAgent(fn=my_func_original, name="my_func")
```

In [ ]:
# @agent decorator example
# After decoration, 'normalise' IS a FunctionAgent instance, not a coroutine function.

@agent
async def normalise(ctx: dict) -> dict:
    """Strip whitespace and lowercase the 'text' field."""
    raw = ctx.get("text", "")
    return {"text": raw.strip().lower()}

# Confirm it is a FunctionAgent, not a plain coroutine
print(f"Type of 'normalise': {type(normalise).__name__}")  # FunctionAgent
print(f"Agent name: {normalise.name}")                      # normalise

# Run it
result = await normalise.run({"text": "   Hello World   "})
print("Decorator agent output:", result)
# Expected: {'text': 'hello world'}

In [ ]:
# You can also pass description and tags to the decorator
@agent(description="Tokenises text into a list of word tokens.", tags=["nlp", "preprocessing"])
async def tokenise(ctx: dict) -> dict:
    """Split normalised text into tokens."""
    text = ctx.get("text", "")
    return {"tokens": text.split()}

print(f"Agent name: {tokenise.name}")
print(f"Description: {tokenise.description}")

result = await tokenise.run({"text": "hello world foo bar"})
print("Tokenise output:", result)
# Expected: {'tokens': ['hello', 'world', 'foo', 'bar']}

## 4. Agent Stats — Inspecting Runtime Metrics

Every `BaseAgent` instance automatically tracks three runtime statistics:

| Field | Meaning |
|---|---|
| `stats.calls` | Total number of times `.run()` has been invoked |
| `stats.errors` | Number of invocations that raised an exception |
| `stats.avg_ms` | Exponential moving average of latency in milliseconds |

These stats are useful for:
- **Debugging**: seeing which agent is called most
- **Performance tuning**: identifying the slowest agent in a chain
- **Reliability monitoring**: spotting agents with high error rates

Stats are attached to the *instance*, so two agents wrapping the same function have independent stats.

In [ ]:
# Build a simple agent that does a small amount of work
@agent
async def slow_square(ctx: dict) -> dict:
    """Simulate a moderately slow computation."""
    await asyncio.sleep(0.02)  # 20 ms simulated latency
    n = ctx.get("n", 0)
    return {"squared": n ** 2}

# Call it 5 times with different inputs
for i in range(1, 6):
    await slow_square.run({"n": i})

# Inspect stats after the calls
s = slow_square.stats
print(f"calls   : {s.calls}")     # 5
print(f"errors  : {s.errors}")    # 0
print(f"avg_ms  : {s.avg_ms:.2f} ms")  # ~20 ms

In [ ]:
# Stats also count errors. Let's build a faulty agent and trigger it.
@agent
async def divide(ctx: dict) -> dict:
    """Divide 100 by ctx['divisor']. Raises ZeroDivisionError if divisor == 0."""
    return {"result": 100 / ctx["divisor"]}

# Two successful calls
await divide.run({"divisor": 4})
await divide.run({"divisor": 5})

# One failing call — the agent catches and re-raises but still increments stats.errors
try:
    await divide.run({"divisor": 0})
except ZeroDivisionError:
    pass  # expected

s = divide.stats
print(f"calls   : {s.calls}")   # 3
print(f"errors  : {s.errors}")  # 1
print(f"avg_ms  : {s.avg_ms:.2f} ms")

## 5. FilterAgent — Gating Data Through a Predicate

`FilterAgent` wraps another agent and only calls it when a **predicate** returns `True`.
When the predicate returns `False`, the agent short-circuits and returns `{}` (an empty delta),
leaving the context unchanged.

This is the Multigen equivalent of an `if` statement in a pipeline. Rather than littering your
agents with conditional logic, you lift the condition to the orchestration layer — keeping
agent code pure and reusable.

### Why not just use an `if` inside the agent?

You could, but `FilterAgent` makes the condition **visible** to the orchestrator. That means:
- Observability tools can log "step skipped" rather than silently passing through
- The predicate can be swapped at runtime without touching agent code
- Stats are still tracked (calls, skipped count) for monitoring

```python
FilterAgent(
    agent=inner_agent,
    predicate=lambda ctx: bool(ctx.get("text")),  # skip if text is empty
    name="filter_nonempty",
)
```

In [ ]:
# Inner agent that does the real work — assume text is already validated
@agent
async def uppercase(ctx: dict) -> dict:
    """Uppercase the text field."""
    return {"text": ctx["text"].upper()}

# Wrap it in a FilterAgent that skips when text is empty or missing
safe_uppercase = FilterAgent(
    agent=uppercase,
    predicate=lambda ctx: bool(ctx.get("text", "").strip()),
    name="safe_uppercase",
)

# Case 1: predicate is True — inner agent runs
result1 = await safe_uppercase.run({"text": "hello world"})
print("Non-empty text:", result1)
# Expected: {'text': 'HELLO WORLD'}

# Case 2: predicate is False — inner agent is skipped, empty dict returned
result2 = await safe_uppercase.run({"text": ""})
print("Empty text (skipped):", result2)
# Expected: {}

In [ ]:
# More realistic example: only run expensive analysis on long documents
@agent
async def expensive_nlp(ctx: dict) -> dict:
    """Simulate a slow NLP pipeline (entity extraction, etc.)."""
    await asyncio.sleep(0.1)  # 100 ms simulated
    return {"entities": ["MOCK_ENTITY_1", "MOCK_ENTITY_2"]}

# Only run NLP on documents with more than 50 words — saves cost on tiny snippets
long_doc_nlp = FilterAgent(
    agent=expensive_nlp,
    predicate=lambda ctx: len(ctx.get("text", "").split()) > 50,
    name="long_doc_nlp",
)

short_text = {"text": "Short doc."}
long_text  = {"text": ("word " * 60).strip()}  # 60 words

r1 = await long_doc_nlp.run(short_text)
r2 = await long_doc_nlp.run(long_text)

print("Short doc result (skipped):", r1)   # {}
print("Long doc result (ran):", r2)         # {'entities': [...]}

## 6. TransformAgent — Pre-Processing Context Before Downstream Agents

`TransformAgent` mutates (or augments) the context *before* passing it to an inner agent.
Think of it as a **lens** or **adapter**: it reshapes the context so the inner agent sees
the exact shape it expects, without the inner agent needing to know where the data came from.

This is extremely useful for:
- **Schema adaptation**: the pipeline uses one key name, the agent uses another
- **Data enrichment**: adding computed fields before the agent runs
- **Normalisation**: stripping HTML, lowercasing, etc. centrally

```python
TransformAgent(
    agent=inner_agent,
    transform_fn=lambda ctx: {**ctx, "text": ctx["raw_text"].lower()},
    name="normalise_then_analyse",
)
```

The `transform_fn` receives the incoming context and returns a **new context dict**.
The inner agent receives this transformed context, not the original.

In [ ]:
# Inner agent expects a 'query' key
@agent
async def search(ctx: dict) -> dict:
    """Mock search agent. Expects ctx['query'] as a list of tokens."""
    tokens = ctx.get("query", [])
    # Simulate returning mock search results for each token
    return {"results": [f"result_for_{t}" for t in tokens]}

# But our pipeline delivers data as ctx['raw_text'] (a string).
# TransformAgent bridges the gap without modifying either the pipeline or the agent.
adapted_search = TransformAgent(
    agent=search,
    transform_fn=lambda ctx: {
        **ctx,
        "query": ctx.get("raw_text", "").lower().split()
    },
    name="text_to_tokens_search",
)

# Pipeline delivers raw_text; the TransformAgent converts it to tokens for the search agent
result = await adapted_search.run({"raw_text": "Machine Learning Transformers"})
print("Search results:", result)
# Expected: {'results': ['result_for_machine', 'result_for_learning', 'result_for_transformers']}

In [ ]:
# Another example: enriching context with computed metadata before passing downstream
@agent
async def summarise(ctx: dict) -> dict:
    """Mock summariser. Uses ctx['char_count'] to decide summary length."""
    char_count = ctx.get("char_count", 0)
    text = ctx.get("text", "")
    # Short summary for short text, longer for long text
    summary_len = min(char_count // 10, 100)
    return {"summary": text[:summary_len] + "..." if len(text) > summary_len else text}

# TransformAgent injects char_count before summarise sees the context
enrich_then_summarise = TransformAgent(
    agent=summarise,
    transform_fn=lambda ctx: {**ctx, "char_count": len(ctx.get("text", ""))},
    name="enrich_summarise",
)

text = "The quick brown fox jumps over the lazy dog. " * 3
result = await enrich_then_summarise.run({"text": text})
print("Summary:", result["summary"])

## 7. RouterAgent — Conditional Dispatch to One of N Agents

`RouterAgent` implements **content-based routing**: it inspects the context, picks exactly one
downstream agent to invoke, and returns that agent's output.

This is the Multigen equivalent of a `switch/case` or `if/elif/else` chain, but at the
orchestration level. The routing logic is encapsulated in a **classifier function** that
maps the context to an agent name (string).

```python
RouterAgent(
    routes={"name": agent_instance, ...},
    classifier=lambda ctx: "name",
    default="fallback_name",   # used if classifier returns unknown key
    name="my_router",
)
```

### Why a classifier function instead of chained FilterAgents?

- **Mutual exclusivity is guaranteed** — only one route runs, no accidental fan-out
- **The classifier is testable in isolation** — unit test the routing logic separately
- **Easy to extend** — add a new route by adding one key to the dict and one `elif` in the classifier

In [ ]:
# Define three specialist agents — each handles one document type
@agent
async def handle_invoice(ctx: dict) -> dict:
    """Extract invoice fields: amount, vendor, date."""
    return {"extracted": {"type": "invoice", "amount": "$1,200", "vendor": "Acme Corp"}}

@agent
async def handle_contract(ctx: dict) -> dict:
    """Extract contract clauses and parties."""
    return {"extracted": {"type": "contract", "parties": ["Party A", "Party B"], "clauses": 12}}

@agent
async def handle_unknown(ctx: dict) -> dict:
    """Fallback handler for unrecognised document types."""
    return {"extracted": {"type": "unknown", "raw": ctx.get("text", "")[:100]}}

# The classifier maps document type to route name
def document_classifier(ctx: dict) -> str:
    """Inspect the first 30 chars of text to decide the document type."""
    text = ctx.get("text", "").lower()
    if "invoice" in text or "amount due" in text:
        return "invoice"      # must match a key in routes
    elif "agreement" in text or "contract" in text:
        return "contract"
    else:
        return "unknown"

# Assemble the router with 3 named routes
doc_router = RouterAgent(
    routes={
        "invoice":  handle_invoice,
        "contract": handle_contract,
        "unknown":  handle_unknown,
    },
    classifier=document_classifier,
    default="unknown",  # safety net if classifier returns an unexpected key
    name="document_router",
)

print("Routes:", list(doc_router.routes.keys()))

In [ ]:
# Test the router with three different inputs

r1 = await doc_router.run({"text": "Invoice #1042 — Amount Due: $1,200 from Acme Corp"})
print("Invoice doc:", r1)
# Expected: extracted type = invoice

r2 = await doc_router.run({"text": "Service Agreement between Party A and Party B dated 2024"})
print("Contract doc:", r2)
# Expected: extracted type = contract

r3 = await doc_router.run({"text": "Random document about cats and dogs."})
print("Unknown doc:", r3)
# Expected: extracted type = unknown

## 8. AggregatorAgent — Parallel Fan-Out with Custom Merge

`AggregatorAgent` runs a **collection of agents in parallel** and then calls a `merge_fn`
to combine their outputs into a single dict.

This is the "gather" pattern: you broadcast the same context to N agents simultaneously,
then reduce their independent results into one coherent output. Total latency is
`max(individual_latencies)` rather than `sum(individual_latencies)`.

```python
AggregatorAgent(
    agents=[agent_a, agent_b, agent_c],
    merge_fn=lambda results: {"merged": results},  # results is a list of dicts
    name="my_aggregator",
)
```

The `merge_fn` receives a **list of dicts** in the same order as `agents`. It must return
a single dict that becomes the aggregator's output.

In [ ]:
# Three analysis agents that run in parallel on the same text
@agent
async def lang_detect(ctx: dict) -> dict:
    """Mock language detection."""
    await asyncio.sleep(0.03)  # simulate API latency
    return {"language": "en", "confidence": 0.98}

@agent
async def sentiment_score(ctx: dict) -> dict:
    """Mock sentiment scoring (returns a float 0..1)."""
    await asyncio.sleep(0.05)  # slightly slower
    text = ctx.get("text", "")
    score = 0.8 if "good" in text.lower() else 0.5
    return {"sentiment_score": score}

@agent
async def keyword_extract(ctx: dict) -> dict:
    """Mock keyword extraction — top 3 non-stopword tokens."""
    await asyncio.sleep(0.04)
    stopwords = {"the", "a", "is", "it", "in", "on", "at", "of", "and"}
    tokens = [w.lower() for w in ctx.get("text", "").split() if w.lower() not in stopwords]
    return {"keywords": tokens[:3]}

# Custom merge function: combine all three analysis results into one dict
def merge_analyses(results: list) -> dict:
    """
    results[0] = lang_detect output
    results[1] = sentiment_score output
    results[2] = keyword_extract output
    """
    merged = {}
    for r in results:
        merged.update(r)  # flatten all keys into one dict
    return merged

text_analyser = AggregatorAgent(
    agents=[lang_detect, sentiment_score, keyword_extract],
    merge_fn=merge_analyses,
    name="text_analyser",
)

start = time.perf_counter()
result = await text_analyser.run({"text": "This is a good day for machine learning."})
elapsed = time.perf_counter() - start

print("Aggregated output:", result)
print(f"Wall time: {elapsed*1000:.1f} ms  (agents ran in parallel, max latency ~50 ms)")
# Expected wall time: ~50 ms (not 30+50+40=120 ms)
# Expected output: {'language': 'en', 'confidence': 0.98, 'sentiment_score': 0.8, 'keywords': [...]}

In [ ]:
# More sophisticated merge: weighted voting / consensus
@agent
async def classifier_model_a(ctx: dict) -> dict:
    await asyncio.sleep(0.02)
    return {"label": "spam", "prob": 0.9}

@agent
async def classifier_model_b(ctx: dict) -> dict:
    await asyncio.sleep(0.03)
    return {"label": "spam", "prob": 0.7}

@agent
async def classifier_model_c(ctx: dict) -> dict:
    await asyncio.sleep(0.01)
    return {"label": "ham", "prob": 0.6}

def majority_vote(results: list) -> dict:
    """Return the label with the highest combined probability across models."""
    scores = {}
    for r in results:
        label = r["label"]
        scores[label] = scores.get(label, 0) + r["prob"]
    winner = max(scores, key=scores.get)
    return {"final_label": winner, "score_breakdown": scores}

ensemble = AggregatorAgent(
    agents=[classifier_model_a, classifier_model_b, classifier_model_c],
    merge_fn=majority_vote,
    name="ensemble_classifier",
)

result = await ensemble.run({"text": "Buy cheap medicine now!"})
print("Ensemble result:", result)
# spam scores: 0.9 + 0.7 = 1.6,  ham scores: 0.6 → winner: spam

## 9. LLMAgent (Mock) — The LLM Interface

`LLMAgent` is the agent type that wraps a language model call. Its interface is **identical**
to every other agent — `async run(ctx) -> dict` — so it slots into any Chain, Graph, or
Parallel without special treatment.

**Important**: Real `LLMAgent` calls require an `OPENAI_API_KEY` (or equivalent) in the
environment. In this notebook we demonstrate the same interface using a `FunctionAgent` mock
so you can run everything offline.

### LLMAgent constructor

```python
from multigen import LLMAgent

summariser = LLMAgent(
    model="gpt-4o-mini",
    system_prompt="You are a concise summariser. Return JSON: {summary: string}",
    output_key="summary",     # which key in context to write the response to
    input_key="text",         # which context key to use as user message
    temperature=0.3,
    max_tokens=256,
    name="llm_summariser",
)
```

When called, it reads `ctx[input_key]`, sends it to the LLM, parses the response, and
writes the result to `ctx[output_key]`.

Below we build a `FunctionAgent` that **mimics this interface exactly** so all downstream
code is identical.

In [ ]:
# This mock has the SAME external interface as a real LLMAgent.
# Swap it for LLMAgent(model='gpt-4o-mini', ...) when you have an API key.

async def _mock_llm_summarise(ctx: dict) -> dict:
    """Simulates an LLM summarisation call by truncating to first 60 chars."""
    await asyncio.sleep(0.05)  # simulate LLM latency
    text = ctx.get("text", "")
    mock_summary = text[:60].rstrip() + "..." if len(text) > 60 else text
    return {"summary": mock_summary}

# This is the mock. In production, replace with:
# from multigen import LLMAgent
# llm_summariser = LLMAgent(model='gpt-4o-mini', input_key='text', output_key='summary')
llm_summariser = FunctionAgent(
    fn=_mock_llm_summarise,
    name="llm_summariser",
    description="Summarises ctx['text'] and writes result to ctx['summary'].",
)

long_doc = {
    "text": (
        "Artificial intelligence is transforming every industry. "
        "From healthcare to finance, models are being deployed at scale. "
        "The challenge is ensuring safety and alignment with human values."
    )
}

result = await llm_summariser.run(long_doc)
print("Summary:", result["summary"])
print("Stats:", llm_summariser.stats.calls, "calls,", f"{llm_summariser.stats.avg_ms:.1f} ms avg")

## 10. Composing Agents — Sequential Calls in Pure Python

Before reaching for `Chain` or `Graph`, it's worth understanding that you can compose agents
with nothing more than a loop and a dict. This is exactly what `Chain` does internally.

Understanding this "bare metal" composition makes it much easier to reason about what
higher-level constructs are doing behind the scenes.

### The pattern

```
ctx = initial_context
for agent in pipeline:
    delta = await agent.run(ctx)
    ctx = {**ctx, **delta}   # merge delta into context
```

Each step *enriches* the context. Later steps can see all keys added by earlier steps.
This is called **context accumulation**.

In [ ]:
# Build a 4-step mini-pipeline manually:
# raw_text → normalise → tokenise → keyword_extract → sentiment_score

pipeline_steps = [normalise, tokenise, keyword_extract, sentiment_score]

# Start with just raw text
ctx = {"text": "  Machine Learning is GREAT for data science!  "}

print("Initial context:", ctx)
print("─" * 50)

for step in pipeline_steps:
    delta = await step.run(ctx)
    ctx = {**ctx, **delta}  # merge: new keys appear, existing keys may be updated
    print(f"After '{step.name}': keys = {list(ctx.keys())}")

print("─" * 50)
print("Final context:")
for k, v in ctx.items():
    print(f"  {k}: {v}")

In [ ]:
# Composing agents sequentially also works with the __call__ shorthand:
# agent(ctx) == await agent.run(ctx)

ctx2 = {"text": "  Deep learning with transformers is a good approach  "}

# Apply each step, accumulating context
ctx2 = {**ctx2, **await normalise(ctx2)}
ctx2 = {**ctx2, **await tokenise(ctx2)}
ctx2 = {**ctx2, **await sentiment_score(ctx2)}

print("Tokens   :", ctx2["tokens"])
print("Sentiment:", ctx2["sentiment_score"])

## Summary

| Agent Type | Use When |
|---|---|
| `FunctionAgent` | You have a Python function and want it in a pipeline |
| `@agent` decorator | Same as FunctionAgent, cleaner syntax for new code |
| `FilterAgent` | You want to conditionally skip a step based on context |
| `TransformAgent` | You want to reshape context before passing it downstream |
| `RouterAgent` | You want mutually exclusive routing to one of N agents |
| `AggregatorAgent` | You want to broadcast to N agents in parallel and merge |
| `LLMAgent` | You want to call a language model as a step |

All types share the same `async run(ctx) -> dict` interface. This uniformity is the
key design principle: **any agent can go anywhere**. Switch from mock to LLM, or from
FunctionAgent to RouterAgent, without changing any other code in your pipeline.

**Next notebook**: `13_sequential_chains.ipynb` — building multi-step pipelines with
Chain, conditional steps, timeout, middleware, and error handling.